In [1]:
"""
Sapphire price-per-carat model v1 (narrow range by default).

Run from model-training-improved/lower_coverage/ (paste into one notebook cell, or run as a script).
Reads  ../master_sapphire_cleaned.csv   and saves   sapphire_model_v1.joblib   in the current folder.

What it does:
  1. Trains on log(Price_Per_Carat); averages XGBoost + CatBoost + LightGBM
  2. 5-fold CV for stable metrics
  3. Price RANGE at several coverage levels. Default = 50% (narrow). 80% / 90% are wider but safer.
  4. Held-out test on 20% unseen stones (range factors calibrated on training stones only)
  5. Saves the final model (trained on ALL stones)
"""
import os
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostRegressor
warnings.filterwarnings("ignore")

CSV = "master_sapphire_cleaned.csv"
MODEL_FILE = "sapphire_model_v1.joblib"
COVERAGES = (0.5, 0.6, 0.8, 0.9)       # which range widths to support
DEFAULT_COVERAGE = 0.5                 # lower = narrower range, but the real price falls outside it more often

if not os.path.exists(CSV):
    raise FileNotFoundError(f"Can't find {CSV!r}. Current folder is {os.getcwd()!r}. Fix CSV or change the working directory.")

# The 13 giant/extreme stones your earlier outlier step already removed.
OUTLIER_IDS = ["U14204","B17591","B2080","B7376","B7377","B8241","B8029","U14739","Y4008","S10048","S21664","S27197","S22114"]

HUE = {"Blue":"blue","Greenish Blue":"blue_green","Bluish Green":"blue_green","Greyish Blue":"blue_other",
       "Purplish Blue":"blue_other","Bluish Grey":"blue_other","Bluish Purple":"blue_other","Violet":"blue_other",
       "Color Change":"color_change","Yellow":"yellow","Orangish Yellow":"yellow_orange","Yellowish Orange":"yellow_orange",
       "Greenish Yellow":"yellow_green","Yellowish Green":"yellow_green","Yellowish Brown":"yellow_other"}
KEEP_ORIGIN = ["Ceylon (Sri Lanka)","Madagascar","Montana","Tanzania","Thailand","Burma (Myanmar)"]
NUM = ["Weight","Length","Width","Height"]
CAT = ["Color","HueGroup","Intensity","Clarity","Shape","Cut","Treatment","Origin"]

def prepare(df):
    d = df.copy()
    d["HueGroup"] = d["Color"].map(HUE)
    d["Shape"]    = d["Shape"].replace({"Asscher - Octagon": "Asscher"})
    d["Origin"]   = d["Origin"].where(d["Origin"].isin(KEEP_ORIGIN), "Other")
    d["Cut"]      = d["Cut"].where(d["Cut"].isin(CUT_KEEP), "Other")
    return d[NUM + CAT]

raw = pd.read_csv(CSV)
raw = raw[~raw.Item_ID.isin(OUTLIER_IDS)].reset_index(drop=True)
CUT_KEEP = raw["Cut"].value_counts()[lambda s: s >= 30].index.tolist()     # rare cuts -> "Other"
X = prepare(raw); y = raw["Price_Per_Carat"].values; ly = np.log(y)

ONEHOT_COLS = pd.get_dummies(X, columns=[c for c in CAT if c != "HueGroup"], drop_first=False).drop(columns="HueGroup").columns
def onehot(Xd):
    return pd.get_dummies(Xd, columns=[c for c in CAT if c != "HueGroup"], drop_first=False).drop(columns="HueGroup") \
             .reindex(columns=ONEHOT_COLS, fill_value=0).astype(float)
def lgb_frame(Xd):
    Z = Xd.copy()
    for c in CAT: Z[c] = pd.Categorical(Z[c], categories=sorted(X[c].unique()))
    return Z

def make_models():
    return {
      "xgb": xgb.XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=0),
      "cat": CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=6, verbose=0, random_seed=0, cat_features=CAT, allow_writing_files=False),
      "lgb": lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=15, subsample=0.8,
                               subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=0),
    }
def inputs(name, Xd):
    return onehot(Xd) if name == "xgb" else lgb_frame(Xd) if name == "lgb" else Xd

def fit_predict(Xtr, ly_tr, Xte):
    """Fit all 3 models on Xtr and return the averaged LOG prediction for Xte."""
    return np.mean([m.fit(inputs(n, Xtr), ly_tr).predict(inputs(n, Xte)) for n, m in make_models().items()], axis=0)

def factors(log_residuals):
    """Range factor for each coverage level: band = prediction x/÷ factor."""
    return {c: float(np.exp(np.quantile(np.abs(log_residuals), c))) for c in COVERAGES}

# ---------- 5-fold CV (out-of-fold predictions) ----------
oof = np.zeros(len(y))
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    oof[te] = fit_predict(X.iloc[tr], ly[tr], X.iloc[te])
p = np.exp(oof)
print(f"CV  MAE=${mean_absolute_error(y,p):.0f}/ct   R2={r2_score(y,p):.3f}   R2(log)={r2_score(ly,oof):.3f}   "
      f"median abs % error={np.median(np.abs(p-y)/y)*100:.1f}%")

FACTOR = factors(ly - oof)
print("Range rule (prediction x/÷ factor): " + ",  ".join(f"{int(c*100)}% -> {f:.2f}" for c, f in FACTOR.items()))

# ---------- Held-out test: 20% of stones are NEVER seen in training ----------
i_tr, i_te = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
Xtr, Xte, ly_tr = X.iloc[i_tr], X.iloc[i_te], ly[i_tr]
oof_tr = np.zeros(len(i_tr))                                   # factors calibrated on TRAINING stones only
for a, b in KFold(5, shuffle=True, random_state=0).split(Xtr):
    oof_tr[b] = fit_predict(Xtr.iloc[a], ly_tr[a], Xtr.iloc[b])
F_tr = factors(ly_tr - oof_tr)
pred = np.exp(fit_predict(Xtr, ly_tr, Xte)); act = y[i_te]
print(f"\nHELD-OUT TEST on {len(i_te)} unseen stones:  MAE=${mean_absolute_error(act,pred):.0f}/ct   R2={r2_score(act,pred):.3f}   "
      f"R2(log)={r2_score(np.log(act),np.log(pred)):.3f}   median abs % error={np.median(np.abs(pred-act)/act)*100:.1f}%")
for c, f in F_tr.items():
    hit = np.mean((act >= pred / f) & (act <= pred * f)) * 100
    print(f"  {int(c*100)}% range (x/÷ {f:.2f}): really contained the actual price for {hit:.1f}% of unseen stones")

# ---------- Final models on ALL stones + predict_range() ----------
FINAL = {n: m.fit(inputs(n, X), ly) for n, m in make_models().items()}

def predict_range(stones: pd.DataFrame, coverage=DEFAULT_COVERAGE):
    """stones: DataFrame with Weight, Length, Width, Height, Color, Intensity, Clarity, Shape, Cut, Treatment, Origin.
    Returns price PER CARAT (low / mid / high) and total_mid = mid x weight."""
    Xn = prepare(stones)
    mid = np.exp(np.mean([m.predict(inputs(n, Xn)) for n, m in FINAL.items()], axis=0))
    out = pd.DataFrame({"low": mid / FACTOR[coverage], "mid": mid, "high": mid * FACTOR[coverage]}).round(0)
    out["total_mid"] = (out["mid"] * stones["Weight"].values).round(0)
    return out

# ---------- Save ----------
import joblib
joblib.dump({"models": FINAL, "factor": FACTOR, "default_coverage": DEFAULT_COVERAGE, "onehot_cols": list(ONEHOT_COLS),
             "cat_levels": {c: sorted(X[c].unique()) for c in CAT}, "cut_keep": CUT_KEEP, "hue": HUE,
             "keep_origin": KEEP_ORIGIN, "num": NUM, "cat": CAT}, MODEL_FILE)
print(f"\nSaved {MODEL_FILE} in {os.getcwd()}")

# ---------- Example: price a new stone ----------
new = pd.DataFrame([{"Weight": 2.10, "Length": 8.4, "Width": 6.9, "Height": 4.6, "Color": "Blue", "Intensity": "Vivid",
                     "Clarity": "Eye Clean", "Shape": "Oval", "Cut": "Mixed Brilliant Cut", "Treatment": "Heat Treated",
                     "Origin": "Ceylon (Sri Lanka)"}])
for c in (0.5, 0.8):
    print(f"\nExample stone, {int(c*100)}% range (per carat):"); print(predict_range(new, coverage=c).to_string(index=False))

CV  MAE=$406/ct   R2=0.793   R2(log)=0.822   median abs % error=22.3%
Range rule (prediction x/÷ factor): 50% -> 1.25,  60% -> 1.34,  80% -> 1.65,  90% -> 2.01

HELD-OUT TEST on 784 unseen stones:  MAE=$424/ct   R2=0.785   R2(log)=0.822   median abs % error=23.1%
  50% range (x/÷ 1.26): really contained the actual price for 49.7% of unseen stones
  60% range (x/÷ 1.35): really contained the actual price for 60.6% of unseen stones
  80% range (x/÷ 1.67): really contained the actual price for 81.6% of unseen stones
  90% range (x/÷ 2.01): really contained the actual price for 91.5% of unseen stones

Saved sapphire_model_v1.joblib in d:\A sliit\Research 1.2\hasi\Gemstone_Detect_Value\model-training-improved

Example stone, 50% range (per carat):
   low    mid   high  total_mid
2563.0 3208.0 4014.0     6737.0

Example stone, 80% range (per carat):
   low    mid   high  total_mid
1947.0 3208.0 5285.0     6737.0


In [2]:
"""Load sapphire_model_v1.joblib and predict a price range for new stones.
Keep this file in the same folder as sapphire_model_v1.joblib. Use the same library versions you trained with."""
import joblib, numpy as np, pandas as pd

B = joblib.load("sapphire_model_v1.joblib")
NUM, CAT = B["num"], B["cat"]

def prepare(df):
    d = df.copy()
    d["HueGroup"] = d["Color"].map(B["hue"])
    d["Shape"]    = d["Shape"].replace({"Asscher - Octagon": "Asscher"})
    d["Origin"]   = d["Origin"].where(d["Origin"].isin(B["keep_origin"]), "Other")
    d["Cut"]      = d["Cut"].where(d["Cut"].isin(B["cut_keep"]), "Other")
    return d[NUM + CAT]

def inputs(name, X):
    if name == "xgb":
        Z = pd.get_dummies(X, columns=[c for c in CAT if c != "HueGroup"], drop_first=False).drop(columns="HueGroup")
        return Z.reindex(columns=B["onehot_cols"], fill_value=0).astype(float)
    if name == "lgb":
        Z = X.copy()
        for c in CAT: Z[c] = pd.Categorical(Z[c], categories=B["cat_levels"][c])
        return Z
    return X                                            # catboost reads the text columns directly

def predict_range(stones, coverage=None):
    coverage = B["default_coverage"] if coverage is None else coverage
    X = prepare(stones)
    mid = np.exp(np.mean([m.predict(inputs(n, X)) for n, m in B["models"].items()], axis=0))
    f = B["factor"][coverage]
    out = pd.DataFrame({"low": mid / f, "mid": mid, "high": mid * f}).round(0)
    out["total_mid"] = (out["mid"] * stones["Weight"].values).round(0)
    return out

if __name__ == "__main__":
    new = pd.DataFrame([{"Weight": 2.10, "Length": 8.4, "Width": 6.9, "Height": 4.6, "Color": "Blue", "Intensity": "Vivid",
                         "Clarity": "Eye Clean", "Shape": "Oval", "Cut": "Mixed Brilliant Cut", "Treatment": "Heat Treated",
                         "Origin": "Ceylon (Sri Lanka)"}])
    print(predict_range(new).to_string(index=False))                   # default: narrow (50%)
    print(predict_range(new, coverage=0.8).to_string(index=False))     # wide (80%)

   low    mid   high  total_mid
2563.0 3208.0 4014.0     6737.0
   low    mid   high  total_mid
1947.0 3208.0 5285.0     6737.0
